## Text Embeddings

In [1]:
# Importing useful dependencies
import boto3
import torch
import chromadb
import numpy as np
import torch.nn.functional as F
import open_clip


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\SakuraSnow\AppData\Local\Programs\Python\Python311\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\SakuraSnow\AppData\Local\Programs\Python\Python311\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\SakuraSnow\AppData\Local\Programs\Python\Python311\Lib\site-packages\ipykernel\

AttributeError: _ARRAY_API not found

In [2]:
# Setup S3 client for MinIO (MinIO implements Amazon S3 API)
s3 = boto3.client(
    "s3",
    endpoint_url="http://127.0.0.1:9000", # MinIO API endpoint
    aws_access_key_id="minioadmin", # User name
    aws_secret_access_key="minioadmin", # Password
)

In [3]:
# Connect to the server (Docker Container)
client = chromadb.HttpClient(host="localhost", port=8000)
# Although we set a path for persistent directory when defining the Docker Container
# It actually stores the embeddings inside the container

# We can use the following line to remove all the stored data in a collection
#client.delete_collection(name="texts")

# Create or get the collection named "texts"
collection = client.create_collection(name="texts", get_or_create=True, embedding_function=None)

In [4]:
# Just in case our device has gpu
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load model
model, _, _ = open_clip.create_model_and_transforms("ViT-B-16", pretrained="openai")
tokenizer = open_clip.get_tokenizer("ViT-B-16") # Tokenizer for texts
model.to(device)

C:\Users\SakuraSnow\AppData\Local\Programs\Python\Python311\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-11): 12 x ResidualAttentionBlock(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((768,), eps=1e-05, elementwise_affine

In [14]:
# We can use this function to retrieve an text from our bucket
def get_text(bucket, key):
    resp = s3.get_object(Bucket=bucket, Key=key)
    body = resp["Body"].read()
    text = body.decode("utf-8")
    return text
@torch.no_grad()
# The next function returns the embedding of the given text
def embed_text(tokenizer, model, text):
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    tokens = tokenizer(paragraphs).to(device)

    with torch.no_grad():
        feats = model.encode_text(tokens)
        feats = F.normalize(feats, dim=-1)

    all_embs = feats.detach().cpu().numpy()
    if len(all_embs) == 0:
        return np.zeros(model.config.hidden_size)
    elif len(all_embs) == 1:
        return all_embs[0]
    else:
        full_emb = np.mean(np.stack(all_embs), axis=0)
        full_emb = full_emb / np.linalg.norm(full_emb)
        return full_emb

In [13]:
# The next function stores the embeddings of the texts stored in the Trusted Zone and store them in the collection named 'texts' of our ChromaDB
def texts_to_embeddings(src_bucket, collection, model, tokenizer, src_prefix=""):

    # Incremental id assigned to each embedding
    id_counter = 0
    
    paginator = s3.get_paginator("list_objects_v2") # It returns objects in pages and not all at once.
    for page in paginator.paginate(Bucket=src_bucket, Prefix=src_prefix):

        # List of paths (meta_data)
        file_paths = []
        # List of embeddings
        embeddings = []
        # List of unique IDs for each embedding
        ids = []
        
        for obj in page.get("Contents", []):

            key = obj["Key"]

            if obj['Size'] == 0 and key.endswith("/"): # skip the folder itself
                continue

            id_counter += 1

            # Fetch and open the text file
            response = s3.get_object(Bucket=src_bucket, Key=key)
            body = response["Body"].read().decode("utf-8")
            
            # Compute embedding
            vector = embed_text(tokenizer, model, body) # A numerical vector of size 512

            print(f"Created embedding for {key} ({len(embeddings)} items in current batch).")

            # Storing data
            file_paths.append(f"{src_bucket}/{key}")
            embeddings.append(vector)
            ids.append(f"text_{id_counter}")

        # Store the images of a page at once
        collection.add(
                ids=ids,
                documents=file_paths,
                embeddings=embeddings
        )

        print(f"All embeddings in the current batch are store successfully in the collection {collection.name}.")

In [15]:
texts_to_embeddings(src_bucket = "trusted-zone", src_prefix = "texts/", collection = collection, model = model, tokenizer=tokenizer)

Created embedding for texts/text_1761689048504.txt (0 items in current batch).
Created embedding for texts/text_1761689048588.txt (1 items in current batch).
Created embedding for texts/text_1761689048672.txt (2 items in current batch).
Created embedding for texts/text_1761689048788.txt (3 items in current batch).
Created embedding for texts/text_1761689048863.txt (4 items in current batch).
Created embedding for texts/text_1761689048940.txt (5 items in current batch).
Created embedding for texts/text_1761689049020.txt (6 items in current batch).
Created embedding for texts/text_1761689049098.txt (7 items in current batch).
Created embedding for texts/text_1761689049181.txt (8 items in current batch).
Created embedding for texts/text_1761689049258.txt (9 items in current batch).
Created embedding for texts/text_1761689049336.txt (10 items in current batch).
Created embedding for texts/text_1761689049412.txt (11 items in current batch).
Created embedding for texts/text_1761689049494.txt

In [16]:
# Function that prints the embeddings stored in a collection
def print_stored_embeddings(collection, x=None): # x is the maximum number of files to print
    results = collection.get(include=["documents", "embeddings"])
    for i in range(len(results["documents"])):
        print("ID:", results['ids'][i])
        print("Document:", results["documents"][i])
        print("Embedding (first 5 dims):", results["embeddings"][i][:5])
        print("---")
        if x and (x-1) == i:
            break

# We can use this function to print the embeddings stored in chromaDB
print_stored_embeddings(collection, x = 10)

ID: text_1
Document: trusted-zone/texts/text_1761689048504.txt
Embedding (first 5 dims): [ 0.02143579 -0.05787466  0.02888853 -0.03963237 -0.02045499]
---
ID: text_2
Document: trusted-zone/texts/text_1761689048588.txt
Embedding (first 5 dims): [-0.01391031  0.00854552 -0.02530894  0.05638791 -0.02039851]
---
ID: text_3
Document: trusted-zone/texts/text_1761689048672.txt
Embedding (first 5 dims): [ 0.03187781 -0.06689426  0.00546405  0.05052344 -0.0381616 ]
---
ID: text_4
Document: trusted-zone/texts/text_1761689048788.txt
Embedding (first 5 dims): [ 0.03311309 -0.07672854 -0.02114809 -0.00358424 -0.0640946 ]
---
ID: text_5
Document: trusted-zone/texts/text_1761689048863.txt
Embedding (first 5 dims): [ 0.04894387 -0.05784252  0.02173805 -0.00014183 -0.01029652]
---
ID: text_6
Document: trusted-zone/texts/text_1761689048940.txt
Embedding (first 5 dims): [ 0.0208051  -0.05323555 -0.02754839  0.03454771 -0.0600059 ]
---
ID: text_7
Document: trusted-zone/texts/text_1761689049020.txt
Embeddi

In [17]:
# We can now perform a similarity search to test it

# The following function searches the top k most similar images in ChromaDB using the embeddings of an text
def find_similar_texts(collection, query_emb: np.ndarray, top_k: int = 5):
    # Chroma expects list-of-lists for query_embeddings
    query_vector = query_emb.tolist()

    results = collection.query(
        query_embeddings=[query_vector],
        n_results=top_k,
        include=["documents", "distances"]
    )

    # Extract first query results
    ids = results.get("ids", [[]])[0]
    docs = results.get("documents", [[]])[0]
    dists = results.get("distances", [[]])[0]

    print(f"Top {top_k} similar texts:")
    for rank, (doc_id, doc, dist) in enumerate(zip(ids, docs, dists), start=1):
        similarity = 1 - dist  
        print(f"{rank}. id={doc_id}, distance={dist:.4f} (similarity={similarity:.4f})")
        print(f"   text: {doc[:200]}{'...' if len(doc) > 200 else ''}")

    return results

In [18]:
# Sample query
emb = embed_text(tokenizer, model, "A game similar to Nier: Automata")

# Search for similar texts in ChromaDB
results = find_similar_texts(collection, emb, top_k=10) # The first one is always the target texts itself (if we are using a txt file from MinIO)

Top 10 similar texts:
1. id=text_429, distance=0.4140 (similarity=0.5860)
   text: trusted-zone/texts/text_1761689099393.txt
2. id=text_351, distance=0.4253 (similarity=0.5747)
   text: trusted-zone/texts/text_1761689087075.txt
3. id=text_236, distance=0.4349 (similarity=0.5651)
   text: trusted-zone/texts/text_1761689073176.txt
4. id=text_385, distance=0.4382 (similarity=0.5618)
   text: trusted-zone/texts/text_1761689093538.txt
5. id=text_780, distance=0.4542 (similarity=0.5458)
   text: trusted-zone/texts/text_1761689137917.txt
6. id=text_341, distance=0.4560 (similarity=0.5440)
   text: trusted-zone/texts/text_1761689086158.txt
7. id=text_228, distance=0.4604 (similarity=0.5396)
   text: trusted-zone/texts/text_1761689072296.txt
8. id=text_252, distance=0.4632 (similarity=0.5368)
   text: trusted-zone/texts/text_1761689075029.txt
9. id=text_379, distance=0.4699 (similarity=0.5301)
   text: trusted-zone/texts/text_1761689092162.txt
10. id=text_360, distance=0.4735 (similarity=0.5265